# End-to-end CKI $^{8}$Be: HFB, Gaussian fidelity, symmetry projection, and non-Gaussianity

This notebook follows one nucleus through the complete workflow:

1. read the CKI nuclear-shell-model interaction;
2. build and diagonalize the exact fixed-$(N,Z)$ Hamiltonian;
3. optimize the intrinsic Bogoliubov/HFB energy;
4. find the pure Gaussian state with the largest ground-state fidelity;
5. apply $P_NP_Z$ and then $P_NP_ZP_{J=0}$;
6. compare energies and exact-ground-state fidelities;
7. align the HFB and fidelity-optimized Gaussian by a spatial rotation;
8. measure best-found geometric non-Gaussianity;
9. inspect how the coherent Euler-vacuum series builds the projected state.

The notebook distinguishes an exact statement from a numerical one: every individual rotated Bogoliubov vacuum is Gaussian, but a coherent sum of such vacua is generally non-Gaussian. Non-Gaussianity is nonlinear and cannot be assigned additively to individual series terms.

## 0. Configuration

The checked-in HFB state is loaded by default so the notebook is reproducible and reasonably quick. Set `RECOMPUTE_HFB=True` to repeat the constrained optimization. Set `FULL_GAUSSIAN_SEARCH=True` for both the finite-Thouless and Slater-boundary searches. The latter is recommended for a publication run but is slower.

In [ ]:
RECOMPUTE_HFB = False
FULL_GAUSSIAN_SEARCH = False
RUN_CUMULATIVE_NONGAUSSIANITY = False

HFB_STARTS = 2
GAUSSIAN_STARTS = 10
CUMULATIVE_STARTS = 2


In [ ]:
from pathlib import Path
from typing import Callable, ClassVar, Dict, List, Optional, Tuple
import itertools
import json
import sys
import time

import numpy as np
from scipy.optimize import minimize

ROOT = Path.cwd()
if not (ROOT / 'src' / 'NSMFermions').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src' / 'NSMFermions'))
sys.path.insert(0, str(ROOT / 'benchmarks'))

from cki_be8 import legacy_definitions, build_fermionic_hamiltonian
from hfb import HFBHamiltonian, HFBState, BogoliubovVacuumSeries, solve_hfb
from number_projection import (
    exact_ground_state, number_projected_series, projected_series_observables,
)
from angular_momentum import (
    ParticleNumberJ0ProjectedEnergy, single_particle_angular_momentum,
    euler_rotation, project_state_observables,
)
from gaussian_fidelity import maximize_gaussian_fidelity, maximize_slater_fidelity


## 1. Read the nucleus and construct the NSM Hamiltonian

The valence-space Hamiltonian is

$$H=\sum_{ij}h_{ij}c_i^\dagger c_j+\frac14\sum_{ijkl}\bar v_{ijkl}c_i^\dagger c_j^\dagger c_lc_k.$$

For CKI $^{8}$Be, two valence neutrons and two valence protons occupy twelve $p$-shell modes. The exact fixed-$(N,Z)$ space has dimension 225.

In [ ]:
namespace = dict(globals(), trange=range)
legacy_definitions(
    'cg_utils.py',
    ['CG', 'ClebschGordan', 'SelectCG', 'CreateInitialCGList',
     'CalcInitialValues', 'DivCalc', 'CgJM'],
    namespace,
)
legacy_definitions(
    'nuclear_physics_utils.py',
    ['SingleParticleState', 'krond', 'scattering_matrix_reader',
     'compute_nuclear_twobody_matrix', 'get_twobody_nuclearshell_model'],
    namespace,
)

interaction, eps = namespace['get_twobody_nuclearshell_model'](
    str(ROOT / 'data' / 'cki')
)
single_particle = namespace['SingleParticleState'](str(ROOT / 'data' / 'cki'))
state_encoding = single_particle.state_encoding
ham = HFBHamiltonian(np.diag(eps), interaction)
neutron_modes = list(range(6, 12))
proton_modes = list(range(6))
targets = [2, 2]
fermionic = build_fermionic_hamiltonian(interaction, eps, particles=(2, 2))
exact_energy, exact_target = exact_ground_state(fermionic)

print('modes:', len(eps))
print('fixed-(N,Z) dimension:', len(fermionic.occupations))
print('exact ground-state energy:', exact_energy, 'MeV')


## 2. Optimize or load the intrinsic HFB state

HFB minimizes

$$E[\rho,\kappa]=\langle\Phi|H|\Phi\rangle$$

subject to $\langle N\rangle=2$ and $\langle Z\rangle=2$. The returned `HFBResult` contains the state, energy, particle numbers, convergence flag, optimizer attempts, and stationarity diagnostics.

For this bounded CKI run, pairing collapses numerically: $\|\kappa\|\sim10^{-5}$. A nearly singular-$U$ state lies between the finite-$Z$ and exact-Slater numerical charts, so the helper below replaces it by the corresponding occupied natural-orbital Slater determinant when the collapse diagnostics are small. This changes neither the physical interpretation nor the quoted accuracy, and makes projection robust.

In [ ]:
if RECOMPUTE_HFB:
    hfb_result = solve_hfb(
        ham, neutron_modes, targets, starts=HFB_STARTS, seed=8,
        maxiter=120, tolerance=1e-8,
    )
    hfb_raw = hfb_result.state
    print('optimizer converged:', hfb_result.converged)
    print('optimizer attempts:', hfb_result.attempts)
else:
    saved = np.load(ROOT / 'benchmarks' / 'results' / 'cki_be8_state.npz')
    hfb_raw = HFBState(saved['U'], saved['V'])

rho_hfb = (hfb_raw.rho + hfb_raw.rho.conj().T) / 2
rho_idempotency = np.linalg.norm(rho_hfb @ rho_hfb - rho_hfb)
pairing_norm = np.linalg.norm(hfb_raw.kappa)
occupations_rho, natural_orbitals = np.linalg.eigh(rho_hfb)
hfb_orbitals = natural_orbitals[:, -sum(targets):]
hfb_state = HFBState.from_slater(hfb_orbitals)

print('raw HFB energy:', ham.energy(hfb_raw), 'MeV')
print('Slater-chart energy:', ham.energy(hfb_state), 'MeV')
print('||kappa||:', pairing_norm)
print('||rho^2-rho||:', rho_idempotency)
print('Slater canonical error:', hfb_state.canonical_error())


The intrinsic fidelity is the full-Fock-space quantity

$$F_{\rm HFB}=|\langle\Psi_0|\Phi_{\rm HFB}\rangle|^2.$$

Because $|\Psi_0\rangle$ has fixed $N,Z$, this includes the probability that the intrinsic state lies in that sector.

In [ ]:
hfb_fidelity = hfb_state.fixed_sector_fidelity(
    exact_target, fermionic.occupations
)
hfb_NZ_weight = hfb_state.fixed_sector_weight(fermionic.occupations)
print('HFB N,Z weight:', hfb_NZ_weight)
print('HFB ground-state fidelity:', hfb_fidelity)


## 3. Find the closest pure Gaussian to the exact ground state

The geometric Gaussian fidelity is

$$F_G(\Psi_0)=\max_{|\Omega\rangle\in\mathcal G}|\langle\Psi_0|\Omega\rangle|^2,$$

and the geometric non-Gaussianity is

$$\mathcal N_G(\Psi_0)=1-F_G(\Psi_0).$$

The even Gaussian manifold has a finite-Thouless interior and a singular number-conserving Slater boundary. A complete numerical search must examine both. For CKI $^{8}$Be, repeated Slater starts converge to the same value and outperform the finite-$Z$ search, providing strong numerical evidence—not a mathematical global-optimum certificate—that the closest Gaussian is a Slater determinant.

In [ ]:
slater_best = maximize_slater_fidelity(
    fermionic, exact_target, starts=GAUSSIAN_STARTS, seed=42,
    maxiter=1500, gradient_tolerance=2e-7,
    initial_orbitals=hfb_orbitals,
)
gaussian_candidates = [('Slater boundary', slater_best.fidelity)]
interior_best = None
if FULL_GAUSSIAN_SEARCH:
    interior_best = maximize_gaussian_fidelity(
        fermionic, exact_target, starts=16, seed=41, maxiter=2000,
        tolerance=1e-15, gradient_tolerance=2e-6,
    )
    gaussian_candidates.append(('finite-Z interior', interior_best.fidelity))

best_kind, exact_best_gaussian_fidelity = max(
    gaussian_candidates, key=lambda item: item[1]
)
if best_kind == 'Slater boundary':
    best_gaussian_state = HFBState.from_slater(slater_best.orbitals)
else:
    best_gaussian_state = interior_best.state

print('candidates:', gaussian_candidates)
print('selected:', best_kind)
print('best-found Gaussian fidelity:', exact_best_gaussian_fidelity)
print('best-found ground-state non-Gaussianity:',
      1 - exact_best_gaussian_fidelity)


When `FULL_GAUSSIAN_SEARCH=False`, the displayed value is the optimized Slater-boundary result. Set it to `True` before interpreting the result as the best found over both implemented charts.

## 4. Particle-number projection

The double Fourier projector is

$$P_NP_Z=\frac1{(2\pi)^2}\int d\varphi_Nd\varphi_Z\,e^{i\varphi_N(\hat N-N)}e^{i\varphi_Z(\hat Z-Z)}.$$

In the finite space, the integrals are exact discrete sums. The projected ket remains a coherent `BogoliubovVacuumSeries` until observables require determinant amplitudes.

In [ ]:
pn_series_hfb = number_projected_series(hfb_state, fermionic)
pn_hfb = projected_series_observables(pn_series_hfb, fermionic, exact_target)
pn_series_gaussian = number_projected_series(best_gaussian_state, fermionic)
pn_gaussian = projected_series_observables(
    pn_series_gaussian, fermionic, exact_target
)

print('number-grid vacua:', pn_series_hfb.number_of_vacua)
print('P_N P_Z HFB energy/fidelity:', pn_hfb.energy, pn_hfb.fidelity)
print('P_N P_Z best-Gaussian energy/fidelity:',
      pn_gaussian.energy, pn_gaussian.fidelity)


For a number-conserving Slater determinant with no neutron-proton mixing, $P_NP_Z$ changes nothing. For a paired HFB state, number projection usually produces a non-Gaussian coherent sum.

## 5. Simultaneous number and $J=0$ projection

For the $0^+$ ground state,

$$P_0=\frac1{8\pi^2}\int d\alpha\,d\gamma\,d(\cos\beta)\;R(\alpha,\beta,\gamma).$$

The CKI bounds give a $9\times4\times9$ Euler grid. Combined with the $7\times7$ number grid, each projected series contains 15,876 Gaussian vacua.

In [ ]:
j0_projector = ParticleNumberJ0ProjectedEnergy(
    ham, state_encoding, neutron_modes, targets
)
pnj_series_hfb = j0_projector.projected_series(hfb_state)
pnj_hfb = project_state_observables(pnj_series_hfb, fermionic, exact_target)
pnj_series_gaussian = j0_projector.projected_series(best_gaussian_state)
pnj_gaussian = project_state_observables(
    pnj_series_gaussian, fermionic, exact_target
)

print('Euler grid:', pnj_series_hfb.euler_grid)
print('total vacua:', pnj_series_hfb.number_of_vacua)
print('P_N P_Z P_J=0 HFB energy/fidelity:',
      pnj_hfb.energy, pnj_hfb.fidelity)
print('P_N P_Z P_J=0 best-Gaussian energy/fidelity:',
      pnj_gaussian.energy, pnj_gaussian.fidelity)


## 6. Compare every state with the exact ground state

In [ ]:
rows = [
    ('Exact ground state', exact_energy, 1.0, 1),
    ('Intrinsic HFB/Slater', ham.energy(hfb_state), hfb_fidelity, 1),
    ('Closest Gaussian', ham.energy(best_gaussian_state),
     exact_best_gaussian_fidelity, 1),
    ('P_N P_Z HFB', pn_hfb.energy, pn_hfb.fidelity,
     pn_series_hfb.number_of_vacua),
    ('P_N P_Z closest Gaussian', pn_gaussian.energy, pn_gaussian.fidelity,
     pn_series_gaussian.number_of_vacua),
    ('P_N P_Z P_J=0 HFB', pnj_hfb.energy, pnj_hfb.fidelity,
     pnj_series_hfb.number_of_vacua),
    ('P_N P_Z P_J=0 closest Gaussian', pnj_gaussian.energy,
     pnj_gaussian.fidelity, pnj_series_gaussian.number_of_vacua),
]
print(f"{'state':38s} {'energy (MeV)':>15s} {'F(exact)':>14s} {'vacua':>9s}")
print('-' * 80)
for name, energy, fidelity, vacua in rows:
    print(f'{name:38s} {energy:15.9f} {fidelity:14.9f} {vacua:9d}')


With the checked-in HFB state and converged Slater-boundary search, the reference run gives $F_{\rm HFB}=0.2018135$, $F_{G}=0.2021472$, $F_{NZJ=0}^{\rm HFB}=0.9954831$, and $F_{NZJ=0}^{G}=0.9865244$. Number projection alone barely changes either nearly number-conserving Slater state.

## 7. Are the HFB and closest-Gaussian states related by a rotation?

For Slater orbital matrices $C_H,C_G$,

$$F(\Omega)=|\det(C_H^\dagger R(\Omega)C_G)|^2.$$

The raw overlap can be nearly zero even when the two occupied subspaces differ almost entirely by orientation. The relevant test is $\max_\Omega F(\Omega)$.

In [ ]:
if best_kind != 'Slater boundary':
    raise RuntimeError('This orbital-determinant alignment cell expects the selected Slater boundary')
gaussian_orbitals = slater_best.orbitals
generators = single_particle_angular_momentum(state_encoding)

def slater_overlap_fidelity(left, right):
    return float(abs(np.linalg.det(left.conj().T @ right))**2)

def rotated_fidelity(angles):
    rotation = euler_rotation(*angles, generators)
    return slater_overlap_fidelity(hfb_orbitals, rotation @ gaussian_orbitals)

rng = np.random.default_rng(18)
fits = [
    minimize(lambda angles: -rotated_fidelity(angles),
             rng.uniform(-np.pi, np.pi, 3), method='BFGS',
             options={'maxiter': 300, 'gtol': 1e-10})
    for _ in range(30)
]
alignment = min(fits, key=lambda fit: fit.fun)
raw_mutual_fidelity = slater_overlap_fidelity(hfb_orbitals, gaussian_orbitals)
aligned_mutual_fidelity = float(-alignment.fun)

print('raw HFB/Gaussian fidelity:', raw_mutual_fidelity)
print('best spatially aligned fidelity:', aligned_mutual_fidelity)
print('Euler angles (radians):', alignment.x)


A value near one after alignment means the two intrinsic states are almost the same deformed Slater shape in different laboratory orientations. It does not mean the exact $J=0$ ground state is Gaussian.

## 8. Best-found non-Gaussianity of the $J=0$-projected state

For any normalized fixed-sector target $|\psi\rangle$, the implemented estimate is

$$\mathcal N_G^{\rm best\ found}(|\psi\rangle)=1-\max(F_{\rm finite-Z},F_{\rm Slater}).$$

The optimization is non-convex, so this is a reproducible best-found value, not a certified global optimum. The helper searches the Slater boundary always and the finite-$Z$ chart when `FULL_GAUSSIAN_SEARCH=True`.

In [ ]:
def best_found_gaussian_fidelity(target_vector, starts=GAUSSIAN_STARTS):
    boundary = maximize_slater_fidelity(
        fermionic, target_vector, starts=starts, seed=52,
        maxiter=1500, gradient_tolerance=2e-7,
    )
    candidates = {'Slater boundary': boundary.fidelity}
    interior = None
    if FULL_GAUSSIAN_SEARCH:
        interior = maximize_gaussian_fidelity(
            fermionic, target_vector, starts=max(4, starts), seed=53,
            maxiter=1500, gradient_tolerance=2e-6,
        )
        candidates['finite-Z interior'] = interior.fidelity
    kind = max(candidates, key=candidates.get)
    return candidates[kind], 1-candidates[kind], kind, candidates

j0_best_fidelity, j0_nongaussianity, j0_best_kind, j0_candidates = (
    best_found_gaussian_fidelity(pnj_hfb.projected_vector)
)
print('J=0 projected candidates:', j0_candidates)
print('selected chart:', j0_best_kind)
print('best-found Gaussian fidelity:', j0_best_fidelity)
print('best-found non-Gaussianity:', j0_nongaussianity)


The intrinsic HFB and closest-Gaussian states have zero non-Gaussianity by construction. A number projection of an exactly number-conserving Slater state also leaves it Gaussian. Angular-momentum projection creates a coherent superposition of differently oriented Slater determinants and can therefore generate substantial non-Gaussianity.

## 9. Resolve the $J=0$ series into Euler-group contributions

Write the projected state as

$$|\Psi\rangle=\sum_{r=1}^{L_\Omega}|C_r\rangle,\qquad |C_r\rangle=\sum_{n,z}w_{nzr}G_{nz}R_r|\Phi\rangle.$$

Each individual $G_{nz}R_r|\Phi\rangle$ is Gaussian. Each $|C_r\rangle$ already contains the complete number projector and is generally non-Gaussian. The vectors interfere, so neither norms nor non-Gaussianity add term by term. We can nevertheless inspect cumulative normalized states

$$|\Psi_K\rangle=\frac{\sum_{r=1}^K|C_r\rangle}{\|\sum_{r=1}^K|C_r\rangle\|}.$$

This is a convergence diagnostic, not a unique decomposition: it depends on the chosen Euler-node order.

In [ ]:
def slater_series_components_by_euler(series, orbitals, occupations):
    occupations = np.asarray(occupations, dtype=int)
    euler_points = int(np.prod(series.euler_grid))
    number_points = int(np.prod(series.number_grid))
    transforms = series.transformations.reshape(
        number_points, euler_points, len(orbitals), len(orbitals)
    )
    weights = series.weights.reshape(number_points, euler_points)
    components = np.zeros((euler_points, len(occupations)), complex)
    for number_index in range(number_points):
        for euler_index in range(euler_points):
            rotated = transforms[number_index, euler_index] @ orbitals
            components[euler_index] += (
                weights[number_index, euler_index]
                * np.linalg.det(rotated[occupations])
            )
    return components

euler_components = slater_series_components_by_euler(
    pnj_series_hfb, hfb_orbitals, fermionic.occupations
)
cumulative = np.cumsum(euler_components, axis=0)
checkpoints = sorted(set([1, 2, 4, 8, 16, 32, 64, 128, len(cumulative)]))
cumulative_rows = []
for count in checkpoints:
    vector = cumulative[count-1]
    norm = float(np.vdot(vector, vector).real)
    normalized = vector / np.sqrt(norm)
    exact_fidelity = float(abs(np.vdot(exact_target, normalized))**2)
    cumulative_rows.append((count, norm, exact_fidelity, normalized))

print(f"{'Euler groups K':>14s} {'unnormalized norm':>20s} {'F(exact)':>14s}")
for count, norm, fidelity, _ in cumulative_rows:
    print(f'{count:14d} {norm:20.10e} {fidelity:14.9f}')
full_cumulative = cumulative[-1] / np.linalg.norm(cumulative[-1])
full_series_match = float(
    abs(np.vdot(pnj_hfb.projected_vector, full_cumulative))**2
)
print('full-series projective fidelity:', full_series_match)


The unnormalized norm is not monotonic because complex amplitudes interfere. The final checkpoint must reproduce the full projected state.

In [ ]:
if RUN_CUMULATIVE_NONGAUSSIANITY:
    print(f"{'K':>6s} {'best Gaussian F':>18s} {'N_G':>14s}")
    # Sparse checkpoints keep the repeated non-convex optimizations manageable.
    selected = cumulative_rows[::max(1, len(cumulative_rows)//4)]
    if selected[-1][0] != cumulative_rows[-1][0]:
        selected.append(cumulative_rows[-1])
    for count, _, _, vector in selected:
        best_f, non_g, _, _ = best_found_gaussian_fidelity(
            vector, starts=CUMULATIVE_STARTS
        )
        print(f'{count:6d} {best_f:18.10f} {non_g:14.10f}')
else:
    print('Set RUN_CUMULATIVE_NONGAUSSIANITY=True to run the optional')
    print('ordering-dependent Gaussian optimization at sparse Euler checkpoints.')


## Interpretation

- HFB energy optimization and Gaussian-fidelity optimization solve different variational problems.
- Similar exact-ground-state fidelities do not imply a large mutual overlap.
- For a $J=0$ target, all spatial orientations of the same intrinsic Gaussian have identical target fidelity.
- Rotation alignment tests whether two intrinsic solutions lie on approximately the same symmetry orbit.
- Number and angular-momentum projection restore symmetries by coherent group averaging.
- A single rotated vacuum has $\mathcal N_G=0$, while the symmetry-restored coherent sum may have $\mathcal N_G>0$.
- There is no basis-independent additive 'non-Gaussianity of term $q$'. Cumulative values are useful convergence diagnostics but depend on term grouping and ordering.